In [2]:
import json
import re
import unicodedata
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from jiwer import wer

In [3]:
# ============================================================
# Paths
# ============================================================

TRANSCRIPTION_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\phoasr_val.json")
GROUNDTRUTH_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\demo\transcription\uit_viquad_val.json")
OUTPUT_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA_matched\transcription\audio_context_mapping_phoasr_VIQUAD_val.json")
REJECTED_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA_matched\rejected_mapping_phoasr_VIQUAD_val.json")
TOP_K = 5

Matches audio context files in speech_data with corresponding id field in transcription.json. Done by reading 2 json files and match the closest

In [4]:
MAX_WER = 2

# Number of candidates retrieved by TF-IDF before WER
# verification.
TOP_K = 5


# ============================================================
# Text normalization
# ============================================================

def normalize_text(text):
    """
    Normalize text before matching.

    Vietnamese diacritics are preserved.
    """

    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        r"[^\w\sÀ-ỹ]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# Extract context ID prefix
# ============================================================

def get_context_id_prefix(qa_id):

    parts = qa_id.split("-")

    if len(parts) < 2:
        return None

    return f"{parts[0]}-{parts[1]}"

In [5]:
# ============================================================
# Load files
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcription_data = json.load(f)

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth_data = json.load(f)


# ============================================================
# Extract transcriptions
# ============================================================

transcriptions = [
    item.get("transcription", "")
    for item in transcription_data
]


# ============================================================
# Extract ground-truth contexts
# ============================================================

groundtruth_contexts = []

for data_item in groundtruth_data["data"]:

    for paragraph in data_item["paragraphs"]:

        context = paragraph["context"]

        if context:
            groundtruth_contexts.append(context)


# ============================================================
# Check number of samples
# ============================================================

print(
    f"Transcriptions: {len(transcriptions)}"
)

print(
    f"Ground-truth contexts: {len(groundtruth_contexts)}"
)


if len(transcriptions) != len(groundtruth_contexts):

    raise ValueError(
        "The two files contain different numbers of contexts."
    )


# ============================================================
# Calculate WER
# ============================================================

wers = []

for groundtruth, transcription in zip(
    groundtruth_contexts,
    transcriptions
):

    normalized_groundtruth = normalize_text(
        groundtruth
    )

    normalized_transcription = normalize_text(
        transcription
    )

    if not normalized_transcription:

        continue

    sample_wer = wer(
        normalized_groundtruth,
        normalized_transcription
    )

    wers.append(float(sample_wer))


# ============================================================
# Overall WER
# ============================================================

overall_wer = wer(
    " ".join(
        normalize_text(x)
        for x in groundtruth_contexts
    ),
    " ".join(
        normalize_text(x)
        for x in transcriptions
    )
)


# ============================================================
# Results
# ============================================================

print()
print("============================================================")
print("WER Results")
print("============================================================")

print(
    f"Number of contexts: {len(groundtruth_contexts)}"
)

print(
    f"Contexts evaluated: {len(wers)}"
)

print(
    f"Average context WER: {sum(wers) / len(wers):.4f}"
)

print(
    f"Overall WER: {overall_wer:.4f}"
)

print(
    f"Overall WER (%): {overall_wer * 100:.2f}%"
)


Transcriptions: 557
Ground-truth contexts: 557

WER Results
Number of contexts: 557
Contexts evaluated: 557
Average context WER: 1.2745
Overall WER: 1.0678
Overall WER (%): 106.78%


In [9]:
# ============================================================
# Save WER results to JSON
# ============================================================

WER_OUTPUT_FILE = "wer_results.json"

wer_results = {
    "num_contexts": len(groundtruth_contexts),
    "contexts_evaluated": len(wers),
    "average_context_wer": (
        sum(wers) / len(wers)
        if wers
        else None
    ),
    "overall_wer": float(overall_wer),
    "overall_wer_percent": float(overall_wer * 100),
    "per_context_wer": [
        {
            "file": transcription_data[i]["file"],
            "wer": float(wers[i])
        }
        for i in range(len(wers))
    ]
}

with open(WER_OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        wer_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"\nWER results saved to: {WER_OUTPUT_FILE}"
)



WER results saved to: wer_results.json


In [6]:
import json
import os


# ============================================================
# Load files
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcription_data = json.load(f)

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth_data = json.load(f)


# ============================================================
# Extract ground-truth contexts
# ============================================================

def extract_groundtruth_contexts(data):
    contexts = {}

    for data_item in data["data"]:

        for paragraph in data_item["paragraphs"]:

            context = paragraph["context"]
            qas = paragraph["qas"]

            if not context or not qas:
                continue

            context_id = get_context_id_prefix(qas[0]["id"])

            contexts[context_id] = {
                "context_id": context_id,
                "context": context,
                "qas": qas
            }

    return contexts


groundtruth_contexts = extract_groundtruth_contexts(
    groundtruth_data
)


print(
    f"Loaded {len(groundtruth_contexts)} "
    f"ground-truth contexts"
)


# ============================================================
# Extract ID from transcription filename
# ============================================================

def get_transcription_id(file_name):
    """
    Example:

        ctx_006fe94a.mp3

    ->

        006fe94a
    """

    base_name = os.path.basename(file_name)

    if base_name.startswith("ctx_"):
        base_name = base_name[len("ctx_"):]

    if base_name.endswith(".mp3"):
        base_name = base_name[:-4]

    return base_name


# ============================================================
# Storage
# ============================================================

results = []
rejected_files = []
missing_transcriptions = []


# ============================================================
# Match transcriptions to ground truth
# ============================================================

matched_groundtruth_ids = set()


for transcription_item in transcription_data:

    file_name = transcription_item["file"]
    transcription = transcription_item.get(
        "transcription",
        ""
    )

    transcription_id = get_transcription_id(file_name)


    # --------------------------------------------------------
    # Find matching ground-truth context
    # --------------------------------------------------------

    matching_id = None

    # Direct match
    if transcription_id in groundtruth_contexts:

        matching_id = transcription_id

    else:

        # If GT IDs have an additional suffix/component,
        # try matching based on the first component.
        for context_id in groundtruth_contexts:

            if context_id.split("-")[0] == transcription_id:
                matching_id = context_id
                break


    # --------------------------------------------------------
    # No matching ground truth
    # --------------------------------------------------------

    if matching_id is None:

        rejected_files.append({
            "file": file_name,
            "transcription": transcription,
            "reason": "context_id_not_found_in_groundtruth"
        })

        continue


    groundtruth_item = groundtruth_contexts[matching_id]

    groundtruth = groundtruth_item["context"]

    matched_groundtruth_ids.add(matching_id)


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_transcription = normalize_text(
        transcription
    )

    normalized_groundtruth = normalize_text(
        groundtruth
    )


    # --------------------------------------------------------
    # Empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_files.append({
            "file": file_name,
            "context_id": matching_id,
            "transcription": transcription,
            "matched_context": groundtruth,
            "reason": "empty_transcription"
        })

        continue


    # --------------------------------------------------------
    # Calculate WER
    # --------------------------------------------------------

    context_wer = wer(
        normalized_groundtruth,
        normalized_transcription
    )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = len(results) + len(rejected_files)

    if processed % 10 == 0:

        print(
            f"Processed {processed} transcriptions"
        )


# ============================================================
# Find GT contexts with no transcription
# ============================================================

for context_id, groundtruth_item in groundtruth_contexts.items():

    if context_id not in matched_groundtruth_ids:

        missing_transcriptions.append({
            "context_id": context_id,
            "groundtruth": groundtruth_item["context"],
            "reason": "context_id_not_found_in_transcription"
        })


# ============================================================
# Save accepted matches
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save rejected matches
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save missing transcription IDs
# ============================================================

MISSING_FILE = "missing_transcriptions.json"

with open(MISSING_FILE, "w", encoding="utf-8") as f:

    json.dump(
        missing_transcriptions,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Summary
# ============================================================

print()
print("============================================================")
print("Summary")
print("============================================================")

print(
    f"Transcription records: "
    f"{len(transcription_data)}"
)

print(
    f"Ground-truth contexts: "
    f"{len(groundtruth_contexts)}"
)

print(
    f"Accepted: "
    f"{len(results)}"
)

print(
    f"Rejected: "
    f"{len(rejected_files)}"
)

print(
    f"Missing transcriptions: "
    f"{len(missing_transcriptions)}"
)


Loaded 557 ground-truth contexts

Summary
Transcription records: 557
Ground-truth contexts: 557
Accepted: 0
Rejected: 557
Missing transcriptions: 557


In [5]:
# ============================================================
# Load files
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcription_data = json.load(f)

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth_data = json.load(f)


def extract_contexts_by_id(data):
    """
    Extract contexts from a ViSQA JSON file and index them
    by the context ID derived from the first QA ID.

    Supports both:
        {"data": [...]}
    and:
        [...]
    """

    contexts = {}

    # Handle either JSON structure
    if isinstance(data, dict):
        data_items = data["data"]
    elif isinstance(data, list):
        data_items = data
    else:
        raise TypeError(
            f"Expected dict or list, got {type(data).__name__}"
        )

    for data_item in data_items:

        for paragraph in data_item["paragraphs"]:

            context = paragraph["context"]
            qas = paragraph["qas"]

            if not context or not qas:
                continue

            context_id = get_context_id_prefix(
                qas[0]["id"]
            )

            contexts[context_id] = {
                "context_id": context_id,
                "context": context,
                "qas": qas
            }

    return contexts



transcription_contexts = extract_contexts_by_id(
    transcription_data
)

groundtruth_contexts = extract_contexts_by_id(
    groundtruth_data
)


print(
    f"Loaded {len(transcription_contexts)} "
    f"contexts from transcription file"
)

print(
    f"Loaded {len(groundtruth_contexts)} "
    f"contexts from ground-truth file"
)


# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match contexts by ID
# ============================================================

for context_id, transcription_item in (
    transcription_contexts.items()
):

    # --------------------------------------------------------
    # Check whether matching ground-truth context exists
    # --------------------------------------------------------

    if context_id not in groundtruth_contexts:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription_item["context"],
            "reason": "context_id_not_found_in_groundtruth"
        })

        continue


    groundtruth_item = groundtruth_contexts[context_id]


    # --------------------------------------------------------
    # Get contexts
    # --------------------------------------------------------

    transcription = transcription_item["context"]
    groundtruth = groundtruth_item["context"]


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    normalized_transcription = normalize_text(
        transcription
    )

    normalized_groundtruth = normalize_text(
        groundtruth
    )


    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_files.append({
            "context_id": context_id,
            "transcription": transcription,
            "matched_context": groundtruth,
            "reason": "empty_transcription"
        })

        continue


    # --------------------------------------------------------
    # Calculate WER
    #
    # IMPORTANT:
    # WER(reference, hypothesis)
    #
    # Ground truth = reference
    # Transcription = hypothesis
    # --------------------------------------------------------

    context_wer = wer(
        normalized_groundtruth,
        normalized_transcription
    )


    # --------------------------------------------------------
    # Apply WER cutoff
    # --------------------------------------------------------

    if context_wer > MAX_WER:

        rejected_files.append({
            "context_id": context_id,

            "transcription": transcription,

            "matched_context": groundtruth,

            "wer": float(context_wer),

            "wer_cutoff": MAX_WER,

            "reason": (
                "transcription_exceeds_wer_cutoff"
            )
        })

    else:

        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

        results.append({
            "context_id": context_id,

            "transcription": transcription,

            "matched_context": groundtruth,

            "qas": groundtruth_item["qas"],

            "wer": float(context_wer)
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if len(results) % 10 == 0:

        print(
            f"Processed "
            f"{len(results)} contexts"
        )


# ============================================================
# Check for ground-truth contexts that have no transcription
# ============================================================

missing_transcriptions = []

for context_id in groundtruth_contexts:

    if context_id not in transcription_contexts:

        missing_transcriptions.append({
            "context_id": context_id,
            "groundtruth": (
                groundtruth_contexts[context_id]["context"]
            ),
            "reason": "context_id_not_found_in_transcription"
        })


# ============================================================
# Save accepted matches
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save rejected matches
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save missing transcription IDs
# ============================================================

MISSING_FILE = "missing_transcriptions.json"

with open(MISSING_FILE, "w", encoding="utf-8") as f:

    json.dump(
        missing_transcriptions,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Summary
# ============================================================

print()
print("============================================================")
print("Summary")
print("============================================================")

print(
    f"Transcription contexts: "
    f"{len(transcription_contexts)}"
)

print(
    f"Ground-truth contexts: "
    f"{len(groundtruth_contexts)}"
)

print(
    f"Accepted: "
    f"{len(results)}"
)

print(
    f"Rejected: "
    f"{len(rejected_files)}"
)

print(
    f"Missing transcriptions: "
    f"{len(missing_transcriptions)}"
)

KeyError: 'paragraphs'

In [ ]:
# ============================================================
# Load ASR transcriptions
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcriptions = json.load(f)


# ============================================================
# Load ViSQA JSON
# ============================================================

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth = json.load(f)


# ============================================================
# Extract contexts
# ============================================================

contexts = []

for data_item in groundtruth["data"]:

    for paragraph in data_item["paragraphs"]:

        context = paragraph["context"]
        qas = paragraph["qas"]

        if not context or not qas:
            continue

        # Get context ID from the first QA.
        #
        # Example:
        # 0057-0077-0001
        #       ↓
        # 0057-0077
        context_id = get_context_id_prefix(
            qas[0]["id"]
        )

        contexts.append({
            "context_id": context_id,
            "context": context,
            "qas": qas
        })


print(f"Loaded {len(contexts)} unique contexts")
print(f"Loaded {len(transcriptions)} audio transcriptions")


# ============================================================
# Normalize contexts
# ============================================================

normalized_contexts = [
    normalize_text(item["context"])
    for item in contexts
]


# ============================================================
# Character n-gram TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    analyzer="char",

    # 3-5 character n-grams
    ngram_range=(3, 5),

    min_df=1,

    # Helps reduce the effect of very frequent n-grams
    sublinear_tf=True
)

context_vectors = vectorizer.fit_transform(
    normalized_contexts
)


# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match every audio transcription
# ============================================================

for i, item in enumerate(transcriptions):

    audio_file = item["file"]
    transcription = item["transcription"]

    normalized_transcription = normalize_text(
        transcription
    )

    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "reason": "empty_transcription"
        }

        rejected_files.append(rejected_entry)

        continue


    # --------------------------------------------------------
    # Convert ASR transcription to TF-IDF vector
    # --------------------------------------------------------

    transcription_vector = vectorizer.transform(
        [normalized_transcription]
    )


    # --------------------------------------------------------
    # Calculate cosine similarity against all contexts
    # --------------------------------------------------------

    similarities = cosine_similarity(
        transcription_vector,
        context_vectors
    )[0]


    # --------------------------------------------------------
    # Retrieve top-k candidates
    # --------------------------------------------------------

    top_indices = similarities.argsort()[-TOP_K:][::-1]


    candidates = []


    # --------------------------------------------------------
    # Calculate WER for candidates
    # --------------------------------------------------------

    for idx in top_indices:

        candidate_context = normalized_contexts[idx]

        similarity = similarities[idx]

        candidate_wer = wer(
            candidate_context,
            normalized_transcription
        )

        candidates.append({
            "context_id": contexts[idx]["context_id"],
            "similarity": float(similarity),
            "wer": float(candidate_wer)
        })


    # --------------------------------------------------------
    # Select candidate with lowest WER
    # --------------------------------------------------------

    best_candidate = min(
        candidates,
        key=lambda x: x["wer"]
    )

    best_context_id = best_candidate["context_id"]

    best_context = contexts[
        next(
            idx
            for idx in top_indices
            if contexts[idx]["context_id"] == best_context_id
        )
    ]
    # --------------------------------------------------------
    # Apply WER cutoff
    # --------------------------------------------------------

    if best_candidate["wer"] > MAX_WER:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "matched_context": best_context["context"],
            "best_context_id": best_context_id,

            "best_similarity": (
                best_candidate["similarity"]
            ),

            "best_wer": best_candidate["wer"],

            "wer_cutoff": MAX_WER,

            "reason": "best_candidate_exceeds_wer_cutoff",

            "candidates": candidates
        }

        rejected_files.append(rejected_entry)

    else:

        # ----------------------------------------------------
        # Find matched context
        # ----------------------------------------------------

        best_context = next(
            context
            for context in contexts
            if context["context_id"] == best_context_id
        )


        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

        results.append({
            "audio_file": audio_file,

            "transcription": transcription,

            "context_id": best_context_id,

            "matched_context": best_context["context"],

            "qas": best_context["qas"],

            "similarity": best_candidate["similarity"],

            "wer": best_candidate["wer"],

            "candidates": candidates
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 10 == 0:

        print(
            f"Processed "
            f"{i + 1}/{len(transcriptions)}"
        )


# ============================================================
# Save accepted matches
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save rejected files
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2)

Loaded 557 unique contexts
Loaded 557 audio transcriptions
Processed 10/557
Processed 20/557
Processed 30/557
Processed 40/557
Processed 50/557
Processed 60/557
Processed 70/557
Processed 80/557
Processed 90/557
Processed 100/557
Processed 110/557
Processed 120/557
Processed 130/557
Processed 140/557
Processed 150/557
Processed 160/557
Processed 170/557
Processed 180/557
Processed 190/557
Processed 200/557
Processed 210/557
Processed 220/557
Processed 230/557
Processed 240/557
Processed 250/557
Processed 260/557
Processed 270/557
Processed 280/557
Processed 290/557
Processed 300/557
Processed 310/557
Processed 320/557
Processed 330/557
Processed 340/557
Processed 350/557
Processed 360/557
Processed 370/557
Processed 380/557
Processed 390/557
Processed 400/557
Processed 410/557
Processed 420/557
Processed 430/557
Processed 440/557
Processed 450/557
Processed 460/557
Processed 470/557
Processed 480/557
Processed 490/557
Processed 500/557
Processed 510/557
Processed 520/557
Processed 530/

In [12]:
# ============================================================
# Summary
# ============================================================

total = len(transcriptions)

accepted = len(results)

rejected = len(rejected_files)

acceptance_rate = (
    accepted / total * 100
    if total > 0
    else 0
)

rejection_rate = (
    rejected / total * 100
    if total > 0
    else 0
)


print()
print("=" * 60)
print("MATCHING COMPLETE")
print("=" * 60)

print(f"Total audio files:     {total}")
print(f"Accepted matches:      {accepted}")
print(f"Rejected files:        {rejected}")

print(
    f"Acceptance rate:       "
    f"{acceptance_rate:.2f}%"
)

print(
    f"Rejection rate:        "
    f"{rejection_rate:.2f}%"
)

print()
print(f"WER cutoff:             {MAX_WER:.2f}")
print()
print(f"Accepted output:        {OUTPUT_FILE}")
print(f"Rejected output:        {REJECTED_FILE}")
print("=" * 60)


MATCHING COMPLETE
Total audio files:     557
Accepted matches:      557
Rejected files:        0
Acceptance rate:       100.00%
Rejection rate:        0.00%

WER cutoff:             2.00

Accepted output:        G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA_matched\transcription\audio_context_mapping_phoasr_VIQUAD_val.json
Rejected output:        G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA_matched\rejected_mapping_phoasr_VIQUAD_val.json


Field replacement

In [7]:
import json
import pathlib as Path

file1 = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_vit5_visqa_test_results.json"
file2 = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\wer_calc_VISQA_VIQUAD_val.json"
output_file = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_VISQA_VIQUAD_val.json"

# Load file 1
with open(file1, "r", encoding="utf-8") as f:
    data1 = json.load(f)

# Load file 2
with open(file2, "r", encoding="utf-8") as f:
    data2 = json.load(f)

# Replace transcription in file 2 with vit5_prediction from file 1
for i, result2 in enumerate(data2["results"]):
    result2["transcription"] = data1["results"][i]["vit5_prediction"]

# Save as a new file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data2, f, ensure_ascii=False, indent=2)

print(f"Saved updated file to: {output_file}")

Saved updated file to: G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_VISQA_VIQUAD_val.json
